# 갱상도 음성 통역기 v2 — 데이터셋 + 학습 모델 버전

이 노트북은 코랩에서 프로젝트를 **처음부터 새로 생성**합니다. 기존에 꼬인 파일은 지우고 다시 만듭니다.

구성: 합성 경상도 사투리 병렬 데이터셋 + 빠른 retrieval 학습 모델 + 선택형 Hugging Face ByT5 파인튜닝 + Whisper 음성 인식 + FastAPI + Streamlit.

In [1]:
# 1) 의존성 설치
!pip -q install fastapi==0.115.8 "uvicorn[standard]==0.34.0" pydantic==2.10.6 requests==2.32.3 streamlit==1.41.1 pytest==8.3.4 pandas==2.2.2 numpy==1.26.4
!pip -q install transformers==4.45.2 datasets==3.0.2 accelerate==0.34.2 sentencepiece==0.2.0 soundfile==0.12.1 librosa==0.10.2.post1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.1/343.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
# 2) 프로젝트 파일 생성: 기존 폴더 삭제 후 압축 내장본 추출
import base64, shutil, zipfile
from pathlib import Path

PROJECT_DIR = Path("/content/gyeongsang_voice_translator_TRAINED")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

zip_b64 = "".join([
    '...',
])
zip_path = Path("/content/gyeongsang_voice_translator_TRAINED.zip")
zip_path.write_bytes(base64.b64decode(zip_b64))
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

print("프로젝트 생성 완료:", PROJECT_DIR)
print("파일 수:", len(list(PROJECT_DIR.rglob("*"))))
print("데이터셋 미리보기")
!head -n 8 /content/gyeongsang_voice_translator_TRAINED/data/gyeongsang_train.csv

프로젝트 생성 완료: /content/gyeongsang_voice_translator_TRAINED
파일 수: 71
데이터셋 미리보기
﻿dialect,standard,source
니 지금 뭐하노? 밥은 묵었나?,너 지금 뭐 해? 밥은 먹었어?,curated_combined
가게에 퍼뜩 온나.,가게에 빨리 와.,template_place
이거 마이 맛있다,이거 많이 맛있다,template_adj_nopunct
거 참 희한하네,그거 참 신기하네,curated_nopunct
니 역 오나,너 역 와,template_place_nopunct
니 학교 가노?,너 학교 가?,template_place
니 회사 오나,너 회사 와,template_place_nopunct


In [3]:
# 3) 문법/테스트 검증
import os, sys
from pathlib import Path
PROJECT_DIR = Path("/content/gyeongsang_voice_translator_TRAINED")
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

!python -m compileall -q app frontend tests scripts
!python -m pytest -q

........                                                                 [100%]
8 passed in 0.43s


In [4]:
# 4) 빠른 학습: retrieval baseline 모델 생성 + 데모 확인
# 이 셀은 GPU 없이도 몇 초 안에 끝납니다.
import os, sys
from pathlib import Path
PROJECT_DIR = Path("/content/gyeongsang_voice_translator_TRAINED")
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

!python scripts/train_retrieval_baseline.py
!python scripts/evaluate_retrieval.py
!python scripts/quick_predict.py

retrieval trained: {'num_examples': 679, 'model_dir': 'models/retrieval-gyeongsang'}
{'total': 120, 'exact_match': 0.0083, 'avg_similarity': 0.9067}
------------------------------------------------------------
입력: 니 지금 뭐하노? 밥은 묵었나?
표준어: 너 지금 뭐 해? 밥은 먹었어?
엔진: retrieval_trained 신뢰도: 0.99
------------------------------------------------------------
입력: 퍼뜩 온나
표준어: 빨리 와
엔진: retrieval_trained 신뢰도: 0.99
------------------------------------------------------------
입력: 와 이래 늦었노?
표준어: 왜 이렇게 늦었어?
엔진: retrieval_trained 신뢰도: 0.99
------------------------------------------------------------
입력: 이거 억수로 맛있다 아이가.
표준어: 이거 정말 맛있잖아
엔진: retrieval_trained 신뢰도: 0.99
------------------------------------------------------------
입력: 우짜노 큰일났다.
표준어: 어떡하지 큰일 났다.
엔진: retrieval_trained 신뢰도: 0.99


In [5]:
# 5) 선택: Hugging Face ByT5 파인튜닝
# 시간이 걸립니다. GPU 런타임에서 실행하세요.
# 발표 시간이 급하면 이 셀은 건너뛰고 6번으로 가도 됩니다.

RUN_BYT5_TRAINING = False

if RUN_BYT5_TRAINING:
    import os, sys
    from pathlib import Path
    PROJECT_DIR = Path("/content/gyeongsang_voice_translator_TRAINED")
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, str(PROJECT_DIR))
    !NUM_TRAIN_EPOCHS=3 python scripts/train_byt5.py
else:
    print("ByT5 파인튜닝은 건너뜁니다. 실행하려면 RUN_BYT5_TRAINING = True 로 바꾸세요.")

ByT5 파인튜닝은 건너뜁니다. 실행하려면 RUN_BYT5_TRAINING = True 로 바꾸세요.


In [9]:
# 현재 코랩 프로젝트의 Streamlit headers 오류 즉시 수정

from pathlib import Path
import subprocess
import time

PROJECT_DIR = Path("/content/gyeongsang_voice_translator_TRAINED")
frontend_path = PROJECT_DIR / "frontend" / "app.py"

assert frontend_path.exists(), f"파일을 찾을 수 없습니다: {frontend_path}"

fixed_code = r'''
from __future__ import annotations

import base64
import requests
import streamlit as st

st.set_page_config(page_title="갱상도 음성 통역기", page_icon="🎙️", layout="centered")

st.title("🎙️ 갱상도 음성 통역기")
st.caption("경상도 사투리 음성/텍스트를 표준어로 바꾸는 Hugging Face 기반 AI 서비스")

with st.sidebar:
    st.header("API 설정")
    api_base = st.text_input("FastAPI 주소", value="http://127.0.0.1:8000")
    api_key = st.text_input("API Key", value="test-key-001", type="password")
    st.markdown("---")
    st.write("추천 데모 문장")
    st.code("니 지금 뭐하노? 밥은 묵었나?\n퍼뜩 온나\n와 이래 늦었노?", language="text")

# 핵심 수정: API 요청용 headers를 항상 먼저 정의
api_base = api_base.rstrip("/")
headers = {"X-API-Key": api_key}


def render_result(data: dict):
    st.subheader("결과")

    c1, c2 = st.columns(2)
    with c1:
        st.metric("신뢰도", f"{data.get('confidence', 0) * 100:.1f}%")
    with c2:
        st.metric("엔진", data.get("engine", "unknown"))

    st.write("**인식/입력 문장**")
    st.info(data.get("original_transcript", ""))

    st.write("**표준어 번역**")
    st.success(data.get("standard_korean", ""))

    st.write("**의도**")
    st.write(data.get("intent", ""))

    expressions = data.get("expressions", [])
    if expressions:
        st.write("**감지된 사투리 표현**")
        for item in expressions:
            st.markdown(
                f"- `{item.get('dialect', '')}` → "
                f"**{item.get('standard', '')}**: "
                f"{item.get('description', '')}"
            )

    st.write("**답장 추천**")
    st.write(data.get("reply_suggestion", ""))

    with st.expander("JSON 응답 보기"):
        st.json(data)


tab_text, tab_audio = st.tabs(["텍스트 테스트", "음성 테스트"])

with tab_text:
    st.subheader("텍스트로 먼저 안정성 확인")
    text = st.text_area(
        "경상도 사투리를 입력하세요",
        value="니 지금 뭐하노? 밥은 묵었나?",
        height=120,
    )

    if st.button("텍스트 번역", type="primary"):
        try:
            res = requests.post(
                f"{api_base}/predict-text",
                headers=headers,
                json={"text": text},
                timeout=120,
            )

            if res.status_code != 200:
                st.error(f"API 오류 {res.status_code}: {res.text}")
            else:
                render_result(res.json())

        except Exception as e:
            st.error(f"요청 실패: {e}")

with tab_audio:
    st.subheader("음성 업로드/녹음")
    st.warning("처음 음성 실행은 Whisper 모델 다운로드 때문에 시간이 걸릴 수 있습니다.")

    audio_file = st.file_uploader(
        "오디오 파일 업로드",
        type=["wav", "mp3", "m4a", "ogg", "flac", "webm"],
    )

    recorded_audio = None
    if hasattr(st, "audio_input"):
        recorded_audio = st.audio_input("마이크로 직접 녹음")

    selected = recorded_audio or audio_file

    if selected is not None:
        st.audio(selected)

        if st.button("음성 번역", type="primary"):
            try:
                audio_bytes = selected.getvalue()

                payload = {
                    "audio_base64": base64.b64encode(audio_bytes).decode("utf-8"),
                    "file_name": getattr(selected, "name", "recorded.wav") or "recorded.wav",
                    "content_type": getattr(selected, "type", "audio/wav") or "audio/wav",
                }

                res = requests.post(
                    f"{api_base}/predict",
                    headers=headers,
                    json=payload,
                    timeout=300,
                )

                if res.status_code != 200:
                    st.error(f"API 오류 {res.status_code}: {res.text}")
                else:
                    render_result(res.json())

            except Exception as e:
                st.error(f"요청 실패: {e}")
'''

frontend_path.write_text(fixed_code, encoding="utf-8")

# 기존 서버 종료
subprocess.run("pkill -f 'uvicorn app.main:app' || true", shell=True)
subprocess.run("pkill -f 'streamlit run frontend/app.py' || true", shell=True)
time.sleep(2)

# 문법 확인
subprocess.run(
    f"cd {PROJECT_DIR} && python -m compileall -q frontend/app.py",
    shell=True,
    check=True,
)

print("수정 완료: frontend/app.py")
print("이제 6번 서버 실행 셀을 다시 실행하세요.")

수정 완료: frontend/app.py
이제 6번 서버 실행 셀을 다시 실행하세요.


In [ ]:
# 6) FastAPI + Streamlit 실행 - Colab SyntaxError 완전 수정 버전
# 출력되는 Colab proxy URL을 여세요. 이 셀은 중지하지 말고 유지하세요.

import os
import sys
import time
import subprocess
import urllib.request
from pathlib import Path

PROJECT_DIR = Path("/content/gyeongsang_voice_translator_TRAINED")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

FASTAPI_LOG = "/content/fastapi.log"
STREAMLIT_LOG = "/content/streamlit.log"

print("현재 작업 폴더:", os.getcwd())

# 기존 서버 종료
for cmd in [
    "pkill -f 'uvicorn app.main:app' || true",
    "pkill -f 'streamlit run frontend/app.py' || true",
]:
    subprocess.run(cmd, shell=True)

time.sleep(2)

# 로그 초기화
for log_path in [FASTAPI_LOG, STREAMLIT_LOG]:
    try:
        Path(log_path).unlink()
    except FileNotFoundError:
        pass

# FastAPI 실행
print("FastAPI 서버 실행 중...")
fastapi_out = open(FASTAPI_LOG, "w", encoding="utf-8")
fastapi_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app.main:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
    ],
    cwd=str(PROJECT_DIR),
    stdout=fastapi_out,
    stderr=subprocess.STDOUT,
)

def wait_url(url, timeout=40):
    start = time.time()
    last_error = None

    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.read().decode("utf-8", errors="replace")
        except Exception as exc:
            last_error = exc
            time.sleep(1)

    raise RuntimeError(f"서버 응답 대기 실패: {url} / 마지막 오류: {last_error}")

try:
    health_body = wait_url("http://127.0.0.1:8000/health", timeout=50)
    print("FastAPI health check 성공:")
    print(health_body)
except Exception as exc:
    print("FastAPI 시작 실패:", exc)
    print("아래 7번 로그 확인 셀을 실행하세요.")
    raise

# Streamlit 실행
print("Streamlit 서버 실행 중...")
streamlit_out = open(STREAMLIT_LOG, "w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "frontend/app.py",
        "--server.address",
        "127.0.0.1",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false",
        "--server.fileWatcherType",
        "none",
        "--browser.gatherUsageStats",
        "false",
    ],
    cwd=str(PROJECT_DIR),
    stdout=streamlit_out,
    stderr=subprocess.STDOUT,
)

time.sleep(8)

# Streamlit 접속 링크 출력
print("=" * 80)

try:
    from google.colab.output import eval_js

    url = eval_js("google.colab.kernel.proxyPort(8501)")
    print("Streamlit 접속 링크:")
    print(url)
except Exception as exc:
    print("Colab proxy URL 생성 실패:", exc)
    print("로컬 실행이면 아래 주소로 접속하세요.")
    print("http://127.0.0.1:8501")

print("=" * 80)
print("화면 설정값")
print("FastAPI 주소: http://127.0.0.1:8000")
print("API Key: test-key-001")
print("=" * 80)
print("이 셀은 실행 상태로 두세요. 중지하면 사이트도 꺼집니다.")

while True:
    time.sleep(60)

현재 작업 폴더: /content/gyeongsang_voice_translator_TRAINED
FastAPI 서버 실행 중...
FastAPI health check 성공:
{"status":"ok","app":"gyeongsang_voice_translator","version":"2.0.0"}
Streamlit 서버 실행 중...
Streamlit 접속 링크:
https://8501-gpu-t4-s-kkb-euw4c1-1525lb1b6olgx-c.europe-west4-1.prod.colab.dev
화면 설정값
FastAPI 주소: http://127.0.0.1:8000
API Key: test-key-001
이 셀은 실행 상태로 두세요. 중지하면 사이트도 꺼집니다.


In [ ]:
# 7) 문제가 생겼을 때 로그 확인
from pathlib import Path

def show_log(title, path, n=120):
    print("=" * 80)
    print(title)
    print("=" * 80)
    p = Path(path)
    if not p.exists():
        print("로그 파일 없음:", path)
        return
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    for line in lines[-n:]:
        print(line)

show_log("FastAPI log", "/content/fastapi.log")
show_log("Streamlit log", "/content/streamlit.log")